# 09 · Tabla plana para Tableau

El panel de Tableau es una sola vista, así que no usa el modelo en estrella de `data/modelo/`: lee una tabla
plana con una fila por jugador. Tableau Public no se conecta a un CSV por url; el fichero se carga desde el disco
y se empaqueta al publicar. Por eso `tableau/datos_tableau.csv` está versionado: es el dato de origen del panel.

Genera:

- `tableau/datos_tableau.csv`: 561 jugadores con sus datos, `Percentil fiable`, `Jugadores de referencia` y las
  21 métricas, cada una con su valor y su percentil. Los nombres de columna son los que se ven en Tableau.
- `tableau/campos_calculados.md`: los parámetros y campos calculados del panel, escritos a partir de los
  mismos nombres de columna, para que el código y el CSV no se desajusten nunca.

Un dato que falta se queda vacío, nunca a cero.

In [1]:
# Instalamos las librerías del cuaderno; si ya están instaladas no cambia nada
%pip install -q pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Importamos os para situarnos en la raíz del proyecto
import os
# Importamos pandas para trabajar con tablas
import pandas as pd

# Si el cuaderno se ejecuta desde notebooks/, subimos a la raíz para usar rutas relativas
if os.path.basename(os.getcwd()) == "notebooks":
    # Cambiamos a la carpeta superior
    os.chdir("..")

# Carpeta del modelo y carpeta de salida
MODELO = "data/modelo"
SALIDA = "tableau"
# Nombre para mostrar de cada grupo de posición
NOMBRE_GRUPO = {"central": "Central", "lateral": "Lateral", "pivote_mediocentro": "Pivote y mediocentro",
                "mediapunta": "Mediapunta", "extremo": "Extremo", "delantero": "Delantero",
                "sin_definir": "Sin definir", "sin_confirmar": "Posición no confirmada"}

## 1. Datos del jugador

In [3]:
# Tablas del modelo
jug = pd.read_csv(f"{MODELO}/dim_jugador.csv")
eq = pd.read_csv(f"{MODELO}/dim_equipo.csv")
pos = pd.read_csv(f"{MODELO}/dim_posicion.csv")
hechos = pd.read_csv(f"{MODELO}/hechos_jugador.csv")
metricas = pd.read_csv(f"{MODELO}/dim_metrica.csv").sort_values("id_metrica")
largo = pd.read_csv(f"{MODELO}/hechos_metrica.csv")

# Un grupo sin nombre para mostrar pararía aquí
assert set(pos["grupo"]) <= set(NOMBRE_GRUPO), "falta el nombre de algún grupo de posición"

# Jugador con su equipo, su posición y sus dos columnas de fiabilidad
base = (jug.merge(eq[["id_equipo", "nombre_equipo", "url_escudo"]], on="id_equipo", how="left")
           .merge(pos[["id_posicion", "grupo"]], on="id_posicion", how="left")
           .merge(hechos[["id_jugador", "percentil_fiable", "n_grupo_referencia"]], on="id_jugador", how="left"))
# Columnas con el nombre que se verá en Tableau
datos = pd.DataFrame({
    "ID jugador": base["id_jugador"],
    "Jugador": base["jugador"],
    "Equipo": base["nombre_equipo"],
    "Foto (url)": base["url_foto"],
    "Escudo (url)": base["url_escudo"],
    "Puesto": base["grupo"].map(NOMBRE_GRUPO),
    "Edad": base["edad"].astype("Int64"),
    "Minutos": base["minutos"],
    "Partidos": base["partidos"],
    "Altura (cm)": base["altura_cm"].astype("Int64"),
    "Pie": base["pie"],
    "Nacionalidad": base["nacionalidad"],
    "Segunda nacionalidad": base["nacionalidad_2"],
    "Valor estimado (€)": base["valor_mercado_tm"].astype("Int64"),
    "Percentil fiable": base["percentil_fiable"],
    "Jugadores de referencia": base["n_grupo_referencia"].astype("Int64"),
})
print(datos.shape)

(561, 16)


## 2. Las 21 métricas: valor y percentil

El valor es por 90 minutos, salvo los dos porcentajes de éxito. El percentil es el de `hechos_metrica`: dentro de
su grupo de posición, sin decimales. Lo tienen todos los jugadores con grupo de posición; si `Percentil fiable` es
falso (menos de 600 minutos), no es comparable.

In [4]:
# Una columna de valor y otra de percentil por métrica, en el orden de dim_metrica
for m in metricas.itertuples():
    # Filas de esa métrica, por jugador
    una = largo[largo["id_metrica"] == m.id_metrica].set_index("id_jugador")
    # Valor con tres decimales como mucho (un vacío sigue vacío)
    datos[m.nombre] = datos["ID jugador"].map(una["valor"].round(3))
    # Percentil entero que puede faltar
    datos[f"{m.nombre} (percentil)"] = datos["ID jugador"].map(una["percentil"]).astype("Int64")

# Cuadre: 561 jugadores, 16 columnas de datos y 2 por métrica
assert len(datos) == len(jug) == 561
assert datos.shape[1] == 16 + 2 * len(metricas)
# Nombres de columna únicos
assert datos.columns.is_unique
# Ningún valor se ha rellenado: los vacíos son los mismos que en el modelo
assert sum(datos[m].isna().sum() for m in metricas["nombre"]) == largo["valor"].isna().sum()

# Guardamos en UTF-8 con BOM para que Tableau y Excel lean bien las tildes
os.makedirs(SALIDA, exist_ok=True)
datos.to_csv(f"{SALIDA}/datos_tableau.csv", index=False, encoding="utf-8-sig")
print(f"{SALIDA}/datos_tableau.csv: {datos.shape[0]} filas, {datos.shape[1]} columnas")
print("valores vacíos por métrica:", {m: int(datos[m].isna().sum()) for m in metricas["nombre"] if datos[m].isna().any()})

tableau/datos_tableau.csv: 561 filas, 58 columnas
valores vacíos por métrica: {'Intercepciones': 5, 'Entradas exitosas': 5, '% de entradas exitosas': 40, 'Despejes': 5, 'Pases bloqueados': 5, 'Duelos aéreos ganados': 5, '% de duelos aéreos ganados': 44, 'Entradas intentadas': 5, 'Duelos aéreos disputados': 5, 'Centros': 5, 'Toques en el área': 5}


## 3. Campos calculados para Tableau

Se escriben a partir de los nombres de columna del CSV. Los valores de los parámetros tienen que coincidir letra
por letra con esos nombres, y así coinciden siempre.

In [5]:
# Nombres de las 21 métricas, en orden
nombres = list(metricas["nombre"])


def caso(parametro, sufijo=""):
    # CASE sobre las 21 métricas: devuelve la columna elegida en el parámetro
    lineas = [f"CASE [{parametro}]"]
    for n in nombres:
        lineas.append(f'WHEN "{n}" THEN [{n}{sufijo}]')
    lineas.append("END")
    return "\n".join(lineas)


# Lista de valores de los parámetros, uno por línea
lista = "\n".join(nombres)
# Texto completo del fichero
md = f"""# Campos calculados del panel de Tableau

Generado por `notebooks/09_tabla_tableau.ipynb` a partir de los nombres de columna de `datos_tableau.csv`: si se
cambia una métrica, se vuelve a ejecutar el cuaderno y este fichero cambia con ella.

Cada bloque se pega en **Análisis → Crear campo calculado**, con el nombre que lleva encima.

> **Sin probar en Tableau.** Estos campos están escritos a partir del CSV, pero no se han pegado ni ejecutado en
> Tableau. Si alguno da error al pegarlo, hay que corregirlo aquí, en el cuaderno 09, y volver a generarlo.

## Parámetros

Crear dos parámetros (**clic derecho en el panel de datos → Crear parámetro**):

| Nombre | Tipo de datos | Valores permitidos | Valor actual |
|---|---|---|---|
| `Eje X` | Cadena | Lista | `Intercepciones` |
| `Eje Y` | Cadena | Lista | `Duelos aéreos ganados` |

Lista de valores, igual para los dos (se puede pegar entera con **Agregar desde el portapapeles**):

```
{lista}
```

Después, **clic derecho en cada parámetro → Mostrar parámetro**.

## Valor del eje X

```
{caso("Eje X")}
```

## Valor del eje Y

```
{caso("Eje Y")}
```

## Percentil del eje X

```
{caso("Eje X", " (percentil)")}
```

## Percentil del eje Y

```
{caso("Eje Y", " (percentil)")}
```

## Mediana del eje X

```
{{ FIXED [Puesto] : MEDIAN(IIF([Percentil fiable], [Valor del eje X], NULL)) }}
```

## Mediana del eje Y

```
{{ FIXED [Puesto] : MEDIAN(IIF([Percentil fiable], [Valor del eje Y], NULL)) }}
```

**Por qué FIXED y no WINDOW_MEDIAN.** La línea de referencia tiene que marcar lo mismo contra lo que se calculó el
percentil: el grupo de referencia, que son los jugadores de ese puesto con percentil fiable (600 minutos o más).
Con WINDOW_MEDIAN la mediana sería la de los jugadores que se ven, y se movería al filtrar por edad, equipo o
minutos: un jugador en el percentil 50 dejaría de caer sobre la línea y el panel se contradiría. Con FIXED sobre
`Puesto`, cada jugador lleva la mediana de su grupo de referencia, que cambia al cambiar de puesto y no cambia con
los demás filtros.

Para que eso se cumpla:

- Los filtros de edad, equipo y minutos tienen que ser **filtros normales de dimensión**. Si alguno se añade al
  contexto (**Agregar al contexto**) o se pone como filtro de la fuente de datos o del extracto, se aplica antes
  del FIXED y la mediana vuelve a moverse.
- En la vista tiene que haber **un solo puesto** (filtro o parámetro de selección única): cada puesto tiene su
  mediana, y con varios a la vez no hay una única línea.
- La línea de referencia se hace con **Analítica → Línea de referencia → Valor: Mediana del eje X, agregado
  Promedio**. Dentro de un puesto el valor es el mismo para todos sus jugadores, así que promedio, mínimo o máximo
  dan lo mismo.
- Un jugador sin dato en la métrica no cuenta para la mediana. La mediana del grupo y el percentil 50 no coinciden
  al milímetro: el percentil cuenta quién queda estrictamente por debajo, así que el jugador mediano sale en torno
  al 49-50.

## Cuadrante

```
IF ISNULL([Valor del eje X]) OR ISNULL([Valor del eje Y]) THEN "Sin dato"
ELSEIF [Valor del eje X] >= [Mediana del eje X] AND [Valor del eje Y] >= [Mediana del eje Y]
    THEN "Por encima en " + [Eje X] + " y en " + [Eje Y]
ELSEIF [Valor del eje X] >= [Mediana del eje X]
    THEN "Por encima en " + [Eje X] + ", por debajo en " + [Eje Y]
ELSEIF [Valor del eje Y] >= [Mediana del eje Y]
    THEN "Por debajo en " + [Eje X] + ", por encima en " + [Eje Y]
ELSE "Por debajo en " + [Eje X] + " y en " + [Eje Y]
END
```

Es un cálculo de fila, no de tabla: compara cada jugador con la mediana de su grupo de referencia, la misma de las
líneas, así que tampoco cambia con los filtros de edad, equipo o minutos. Quien está justo en la mediana cuenta
como «por encima».
"""
# Guardamos el fichero
with open(f"{SALIDA}/campos_calculados.md", "w", encoding="utf-8") as f:
    f.write(md)
print(f"{SALIDA}/campos_calculados.md escrito")

tableau/campos_calculados.md escrito


## 4. Comprobaciones

In [6]:
# Releemos el CSV como lo leerá Tableau
d = pd.read_csv(f"{SALIDA}/datos_tableau.csv", encoding="utf-8-sig")
# Cada nombre que usa el CASE existe como columna, con y sin "(percentil)"
faltan = [n for n in nombres if n not in d.columns or f"{n} (percentil)" not in d.columns]
print("métricas del CASE sin columna:", faltan or "ninguna")
# Jugadores con percentil fiable (deben ser 368)
print("con percentil fiable:", int(d["Percentil fiable"].sum()))
# Un ejemplo para comprobar a ojo: Diego Llorente en intercepciones (2,06 por 90 y percentil 99)
print(d.loc[d["Jugador"] == "Diego Llorente", ["Jugador", "Equipo", "Puesto", "Intercepciones", "Intercepciones (percentil)"]].to_string(index=False))

métricas del CASE sin columna: ninguna
con percentil fiable: 368
       Jugador     Equipo  Puesto  Intercepciones  Intercepciones (percentil)
Diego Llorente Real Betis Central           2.064                        99.0
